# AlphaLawVA 판례 데이터 EDA

이 노트북은 수집된 판례 원본과 전처리된 판례 데이터를 비교하면서 살펴보기 위한 분석용 파일입니다.

- `r_data`: 원본 상세 JSON(`local_data/precedents/raw/details`)을 표 형태로 펼친 데이터
- `p_data`: 전처리 JSON(`local_data/precedents/processed/cases`)을 표 형태로 모은 데이터

분석 예시로 사건명, 법원명, 사건종류, 선고연도, 섹션별 글자 수, 수집 검색어 분포를 확인합니다.

In [ ]:
# 필요한 라이브러리를 불러옵니다.
from __future__ import annotations

import json
import os
from collections.abc import Iterable
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib_cache"))

import matplotlib.pyplot as plt
import pandas as pd

# macOS 기준 한글 폰트 설정입니다. 다른 OS에서 깨지면 설치된 한글 폰트명으로 바꾸면 됩니다.
plt.rcParams["font.family"] = ["AppleGothic", "Malgun Gothic", "NanumGothic", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 데이터 경로를 설정합니다.
# 노트북을 프로젝트 루트 또는 precedents 폴더에서 열어도 동작하도록 후보 경로를 같이 봅니다.
def find_project_root() -> Path:
    """현재 위치 주변에서 local_data/precedents 폴더를 가진 프로젝트 루트를 찾습니다."""
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "local_data" / "precedents").exists():
            return candidate.resolve()
    raise FileNotFoundError("local_data/precedents 폴더를 찾지 못했습니다. 프로젝트 안에서 노트북을 열어주세요.")


PROJECT_ROOT = find_project_root()
RAW_DETAILS_DIR = PROJECT_ROOT / "local_data" / "precedents" / "raw" / "details"
PROCESSED_CASES_DIR = PROJECT_ROOT / "local_data" / "precedents" / "processed" / "cases"

# 빠른 테스트만 하고 싶으면 숫자를 넣고, 전체를 보려면 None으로 둡니다.
LIMIT: int | None = None

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DETAILS_DIR:", RAW_DETAILS_DIR)
print("PROCESSED_CASES_DIR:", PROCESSED_CASES_DIR)

In [ ]:
# JSON 파일을 읽고 DataFrame으로 만드는 함수들입니다.
def iter_json_files(directory: Path, limit: int | None = None) -> Iterable[Path]:
    """지정한 폴더의 JSON 파일을 판례일련번호 순서로 반환합니다."""
    paths = sorted(directory.glob("*.json"), key=lambda path: int(path.stem) if path.stem.isdigit() else path.stem)
    if limit is not None:
        paths = paths[:limit]
    yield from paths


def read_json(path: Path) -> dict:
    """UTF-8 JSON 파일 하나를 읽습니다."""
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def flatten_raw_detail(raw: dict, path: Path) -> dict:
    """원본 상세 JSON에서 response.PrecService를 꺼내 표 한 행으로 펼칩니다."""
    service = raw.get("response", {}).get("PrecService", {})
    row = {
        "판례일련번호": str(raw.get("precedent_id") or service.get("판례일련번호") or path.stem),
        "수집시각": raw.get("fetched_at"),
        "매칭검색어": raw.get("matched_queries", []),
        "_파일경로": str(path),
    }
    if isinstance(service, dict):
        row.update(service)
    return row


def load_raw_details(directory: Path, limit: int | None = None) -> pd.DataFrame:
    """원본 상세 판례 JSON들을 읽어 r_data DataFrame으로 만듭니다."""
    rows = [flatten_raw_detail(read_json(path), path) for path in iter_json_files(directory, limit)]
    return pd.DataFrame(rows)


def load_processed_cases(directory: Path, limit: int | None = None) -> pd.DataFrame:
    """전처리된 판례 JSON들을 읽어 p_data DataFrame으로 만듭니다."""
    rows = []
    for path in iter_json_files(directory, limit):
        row = read_json(path)
        row["_파일경로"] = str(path)
        rows.append(row)
    return pd.DataFrame(rows)


def count_empty_fields(data: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """주요 필드별 빈 값 개수와 비율을 계산합니다."""
    total = len(data)
    rows = []
    for column in columns:
        if column not in data.columns:
            continue
        empty_count = data[column].fillna("").astype(str).str.strip().eq("").sum()
        rows.append({"필드": column, "빈값수": int(empty_count), "채워진수": int(total - empty_count), "빈값비율": round(empty_count / total, 4) if total else 0})
    return pd.DataFrame(rows).sort_values("빈값수", ascending=False)

In [ ]:
# 데이터 로드: 이 셀을 실행하면 p_data, r_data가 만들어집니다.
p_data = load_processed_cases(PROCESSED_CASES_DIR, LIMIT)
r_data = load_raw_details(RAW_DETAILS_DIR, LIMIT)

print(f"전처리 데이터 p_data: {p_data.shape[0]:,}행 x {p_data.shape[1]:,}열")
print(f"원본 데이터 r_data: {r_data.shape[0]:,}행 x {r_data.shape[1]:,}열")

In [ ]:
# 전처리 데이터 샘플과 컬럼을 확인합니다.
display(p_data.head(3))
display(pd.DataFrame({"p_data 컬럼": p_data.columns}))

In [ ]:
# 원본 데이터 샘플과 컬럼을 확인합니다.
display(r_data.head(3))
display(pd.DataFrame({"r_data 컬럼": r_data.columns}))

In [ ]:
# 원본과 전처리본의 판례일련번호가 서로 맞는지 확인합니다.
p_ids = set(p_data["판례일련번호"].astype(str))
r_ids = set(r_data["판례일련번호"].astype(str))

print(f"전처리만 있는 ID: {len(p_ids - r_ids):,}개")
print(f"원본만 있는 ID: {len(r_ids - p_ids):,}개")

In [ ]:
# 전처리 데이터의 주요 필드 결측 현황을 봅니다.
MAIN_FIELDS = [
    "사건번호", "접수연도", "사건유형", "접수번호", "사건명", "법원명", "선고일자", "사건종류명", "판결유형",
    "판시사항", "판결요지", "참조조문", "참조판례", "주문", "청구취지", "원심판결", "이유",
]

empty_summary = count_empty_fields(p_data, MAIN_FIELDS)
display(empty_summary)

In [ ]:
# 섹션별 글자 수를 붙이고 분포 요약을 봅니다.
TEXT_FIELDS = ["판시사항", "판결요지", "주문", "청구취지", "원심판결", "이유"]
p_data_len = p_data.copy()

for column in TEXT_FIELDS:
    if column in p_data_len.columns:
        p_data_len[f"{column}_글자수"] = p_data_len[column].fillna("").astype(str).str.len()

length_columns = [f"{column}_글자수" for column in TEXT_FIELDS if f"{column}_글자수" in p_data_len.columns]
display(p_data_len[length_columns].describe().round(1))

In [ ]:
# 사건명별 판례 수 상위 25개를 봅니다.
case_name_counts = p_data["사건명"].fillna("값 없음").astype(str).str.strip().replace("", "값 없음").value_counts().head(25)
display(case_name_counts.to_frame("건수"))

plt.figure(figsize=(10, 7))
case_name_counts.sort_values().plot(kind="barh")
plt.title("사건명별 판례 수 상위 25개")
plt.xlabel("건수")
plt.ylabel("사건명")
plt.tight_layout()
plt.show()

In [ ]:
# 법원명별 판례 수 상위 25개를 봅니다.
court_counts = p_data["법원명"].fillna("값 없음").astype(str).str.strip().replace("", "값 없음").value_counts().head(25)
display(court_counts.to_frame("건수"))

plt.figure(figsize=(10, 7))
court_counts.sort_values().plot(kind="barh")
plt.title("법원명별 판례 수 상위 25개")
plt.xlabel("건수")
plt.ylabel("법원명")
plt.tight_layout()
plt.show()

In [ ]:
# 사건종류명별 판례 수를 봅니다.
case_kind_counts = p_data["사건종류명"].fillna("값 없음").astype(str).str.strip().replace("", "값 없음").value_counts()
display(case_kind_counts.to_frame("건수"))

plt.figure(figsize=(8, 5))
case_kind_counts.sort_values().plot(kind="barh")
plt.title("사건종류명별 판례 수")
plt.xlabel("건수")
plt.ylabel("사건종류명")
plt.tight_layout()
plt.show()

In [ ]:
# 선고연도별 판례 수를 봅니다.
p_data_year = p_data.copy()
p_data_year["선고연도"] = pd.to_datetime(p_data_year["선고일자"], errors="coerce").dt.year
year_counts = p_data_year["선고연도"].value_counts().sort_index()

display(year_counts.tail(20).to_frame("건수"))

plt.figure(figsize=(12, 5))
year_counts.plot(kind="bar")
plt.title("선고연도별 판례 수")
plt.xlabel("선고연도")
plt.ylabel("건수")
plt.tight_layout()
plt.show()

In [ ]:
# 이유 글자 수 분포와 긴 판례 상위 20개를 확인합니다.
display(
    p_data_len[["판례일련번호", "사건번호", "사건명", "법원명", "선고일자", "사건종류명", "이유_글자수"]]
    .sort_values("이유_글자수", ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 5))
plt.hist(p_data_len["이유_글자수"], bins=80)
plt.title("이유 글자 수 분포")
plt.xlabel("이유 글자 수")
plt.ylabel("건수")
plt.tight_layout()
plt.show()

In [ ]:
# 원본 수집 검색어 기준으로 어떤 키워드에서 많이 잡혔는지 확인합니다.
def explode_matched_queries(data: pd.DataFrame) -> pd.DataFrame:
    """원본 matched_queries 배열을 검색어 단위 행으로 펼칩니다."""
    rows = []
    for _, row in data.iterrows():
        for matched in row.get("매칭검색어", []) or []:
            if not isinstance(matched, dict):
                continue
            rows.append({
                "판례일련번호": row["판례일련번호"],
                "검색어": matched.get("query"),
                "검색구분": matched.get("search_type"),
                "search": matched.get("search"),
            })
    return pd.DataFrame(rows)


matched_query_data = explode_matched_queries(r_data)
display(matched_query_data.head())

if not matched_query_data.empty:
    query_counts = matched_query_data["검색어"].fillna("값 없음").value_counts().head(30)
    display(query_counts.to_frame("건수"))

    plt.figure(figsize=(10, 7))
    query_counts.sort_values().plot(kind="barh")
    plt.title("수집 검색어별 매칭 판례 수 상위 30개")
    plt.xlabel("건수")
    plt.ylabel("검색어")
    plt.tight_layout()
    plt.show()

In [ ]:
# 관심 키워드가 사건명과 이유에 얼마나 등장하는지 간단히 봅니다.
KEYWORDS = ["임대차", "전세", "월세", "보증금", "매매", "대항력", "우선변제권", "근저당권", "중개사"]

keyword_rows = []
for keyword in KEYWORDS:
    case_name_count = p_data["사건명"].fillna("").astype(str).str.contains(keyword, regex=False).sum()
    reason_count = p_data["이유"].fillna("").astype(str).str.contains(keyword, regex=False).sum()
    keyword_rows.append({"키워드": keyword, "사건명등장수": int(case_name_count), "이유등장수": int(reason_count)})

keyword_summary = pd.DataFrame(keyword_rows).sort_values("이유등장수", ascending=False)
display(keyword_summary)

## 판시사항/판결요지 없는 판례의 본문 길이 분포

판시사항과 판결요지가 둘 다 비어 있는 판례만 따로 골라서, `판례내용` 원문 길이와 전처리된 `이유` 길이 분포를 확인합니다.


In [ ]:
# 판시사항/판결요지가 둘 다 없는 판례의 판례내용/이유 글자 수 분포를 확인합니다.
# raw 원본의 판례내용 길이와 processed 전처리본의 이유 길이를 나란히 비교합니다.

import numpy as np

# 원본 r_data와 전처리 p_data에 글자 수 컬럼을 추가합니다.
r_len = r_data.copy()
p_len = p_data.copy()

if "판례내용" in r_len.columns:
    r_len["판례내용_글자수"] = r_len["판례내용"].fillna("").astype(str).str.len()
else:
    r_len["판례내용_글자수"] = 0

p_len["판시사항_있음"] = p_len["판시사항"].fillna("").astype(str).str.strip().ne("")
p_len["판결요지_있음"] = p_len["판결요지"].fillna("").astype(str).str.strip().ne("")
p_len["이유_글자수"] = p_len["이유"].fillna("").astype(str).str.len()

missing_issue_summary = p_len[(~p_len["판시사항_있음"]) & (~p_len["판결요지_있음"])].copy()

# raw 원본 판례내용 글자 수를 판례일련번호 기준으로 붙입니다.
if "판례일련번호" in r_len.columns:
    raw_length_cols = r_len[["판례일련번호", "판례내용_글자수"]].copy()
    raw_length_cols["판례일련번호"] = raw_length_cols["판례일련번호"].astype(str)
    missing_issue_summary["판례일련번호"] = missing_issue_summary["판례일련번호"].astype(str)
    missing_issue_summary = missing_issue_summary.merge(raw_length_cols, on="판례일련번호", how="left")
else:
    missing_issue_summary["판례내용_글자수"] = np.nan

print(f"판시사항/판결요지 둘 다 없는 판례 수: {len(missing_issue_summary):,}건")

display(
    missing_issue_summary[[
        "판례일련번호", "사건번호", "사건명", "법원명", "선고일자", "사건종류명", "판례내용_글자수", "이유_글자수"
    ]]
    .sort_values("이유_글자수", ascending=False)
    .head(20)
)

length_bins = [0, 300, 1000, 3000, 5000, 10000, 30000, 50000, 100000, np.inf]
length_labels = ["0~300자", "300~1천자", "1천~3천자", "3천~5천자", "5천~1만자", "1만~3만자", "3만~5만자", "5만~10만자", "10만자 이상"]

reason_bin_counts = (
    pd.cut(missing_issue_summary["이유_글자수"], bins=length_bins, labels=length_labels, include_lowest=True, right=False)
    .value_counts()
    .reindex(length_labels, fill_value=0)
)
reason_bin_table = reason_bin_counts.to_frame("판례수")
reason_bin_table["비율"] = (reason_bin_table["판례수"] / len(missing_issue_summary) * 100).round(2)

display(reason_bin_table)

plt.figure(figsize=(10, 5))
plt.hist(missing_issue_summary["이유_글자수"], bins=80)
plt.title("판시사항/판결요지 없는 판례의 이유 글자 수 분포")
plt.xlabel("이유 글자 수")
plt.ylabel("판례 수")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
reason_bin_counts.plot(kind="bar")
plt.title("판시사항/판결요지 없는 판례의 이유 글자 수 구간별 판례 수")
plt.xlabel("이유 글자 수 구간")
plt.ylabel("판례 수")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

if missing_issue_summary["판례내용_글자수"].notna().any():
    raw_bin_counts = (
        pd.cut(missing_issue_summary["판례내용_글자수"], bins=length_bins, labels=length_labels, include_lowest=True, right=False)
        .value_counts()
        .reindex(length_labels, fill_value=0)
    )
    raw_bin_table = raw_bin_counts.to_frame("판례수")
    raw_bin_table["비율"] = (raw_bin_table["판례수"] / len(missing_issue_summary) * 100).round(2)
    display(raw_bin_table)

    plt.figure(figsize=(10, 5))
    plt.hist(missing_issue_summary["판례내용_글자수"].fillna(0), bins=80)
    plt.title("판시사항/판결요지 없는 판례의 원본 판례내용 글자 수 분포")
    plt.xlabel("판례내용 글자 수")
    plt.ylabel("판례 수")
    plt.tight_layout()
    plt.show()
